# Lab 1B — Refactor the Legacy Script

**Week 1 · Day 1 · AI Engineering Academy — the anchor lab that closes Day 1**

⏱️ **Time-box:** ~70–90 minutes.

## The situation
You've inherited `legacy_report.py` — a working but ugly procedural script that summarizes synthetic **Cordwell Home & Hardware** orders. It has nested copy-paste loops, a **bare `except:` that swallows errors silently**, magic numbers scattered throughout, and zero type hints. Your job is to refactor it into the clean, typed, fail-loud shape every later week builds on — then wire it into the morning's `LLMConfig` + `call_model()` pattern to close the day.

## The plan
| Segment | Focus | ~Time |
|---|---|---|
| Part 0 | Read the legacy script, run the baseline | 5 min |
| Part 1 | Refactor: constants, dataclass, parse, transforms, smoke test | 35–45 min |
| Part 2 | Validation, custom exception, docstrings (stretch) | 15–20 min |
| Part 3 | Pull the day together: config object + model stub | 10–15 min |
| Close | Restart & Run All + checks for understanding | 5 min |

**Total: ~70–90 minutes.**

> 🔁 **House rules:** *fail loudly* (no silent fallbacks) and *Restart-and-Run-All* (a notebook that only works in the order you happened to run cells is not done).

## Part 0 — The starting point

Run the two cells below first. The data cell defines `RAW_ORDERS`; the legacy cell prints the **baseline** your refactor must reproduce. Read the legacy code — do **not** edit it. Notice the four things wrong with it (bare except, copy-paste loops, magic numbers `100`/`500`, no types).

In [ ]:
# Synthetic Cordwell Home & Hardware contractor orders (fictional data).
# This is the raw input both the legacy script and your refactor will read.
RAW_ORDERS: list[dict] = [
    {"id": "ORD-1001", "amount": 150.00,  "status": "shipped",   "customer": "Ridgeline Builders"},
    {"id": "ORD-1002", "amount": 42.50,   "status": "pending",   "customer": "Hammersmith Contracting"},
    {"id": "ORD-1003", "amount": 980.00,  "status": "shipped",   "customer": "Oakfield Renovations"},
    {"id": "ORD-1004", "amount": 18.99,   "status": "shipped",   "customer": "DIY Walk-in"},
    {"id": "ORD-1005", "amount": 305.75,  "status": "cancelled", "customer": "Maple & Stone LLC"},
    {"id": "ORD-1006", "amount": 615.00,  "status": "shipped",   "customer": "Ironclad Decks"},
    {"id": "ORD-1007", "amount": 74.20,   "status": "returned",  "customer": "Hammersmith Contracting"},
    {"id": "ORD-1008", "amount": 1240.00, "status": "shipped",   "customer": "Summit Property Group"},
    {"id": "ORD-1009", "amount": 99.99,   "status": "pending",   "customer": "DIY Walk-in"},
    {"id": "ORD-1010", "amount": 250.00,  "status": "shipped",   "customer": "Ridgeline Builders"},
    {"id": "ORD-1011", "amount": 530.00,  "status": "shipped",   "customer": "Greenfield Landscaping"},
    {"id": "ORD-1012", "amount": 12.00,   "status": "cancelled", "customer": "DIY Walk-in"},
    {"id": "ORD-1013", "amount": 460.00,  "status": "pending",   "customer": "Oakfield Renovations"},
    {"id": "ORD-1014", "amount": 88.50,   "status": "shipped",   "customer": "Maple & Stone LLC"},
]

In [ ]:
# legacy_report.py  --  DO NOT SHIP THIS
# Working but ugly: copy-paste loops, a bare except, magic numbers, zero type hints.
# It runs and prints a baseline. Your refactor must reproduce these numbers exactly.

total = 0
for o in RAW_ORDERS:
    try:
        if o["status"] == "shipped":
            total = total + o["amount"]
    except:
        pass  # <- bare except silently swallows EVERYTHING
legacy_total = total
print("Total shipped: " + str(legacy_total))

# copy-paste block #1 -- magic number 100
flagged = []
for o in RAW_ORDERS:
    if o["amount"] > 100:
        flagged.append(o["id"])
legacy_flagged = flagged
print("High-value order IDs: " + str(legacy_flagged))

# copy-paste block #2 -- manual accumulation
counts = {}
for o in RAW_ORDERS:
    s = o["status"]
    if s in counts:
        counts[s] = counts[s] + 1
    else:
        counts[s] = 1
legacy_counts = counts
print("Counts by status: " + str(legacy_counts))

# copy-paste block #3 -- nested if + another magic number 500
priority = []
for o in RAW_ORDERS:
    if o["status"] == "shipped":
        if o["amount"] > 500:
            priority.append(o["id"])
legacy_priority = priority
print("Priority review IDs: " + str(legacy_priority))

## Part 1 — Refactor

Rebuild the report below. Each section is tagged with the task(s) from the slide it satisfies:

- **Task 01** — Extract typed, single-purpose functions
- **Task 02** — Introduce a dataclass for the order record
- **Task 03** — Replace the bare except; let bugs surface loudly
- **Task 04** — Use a comprehension in place of an accumulator loop
- **Task 05** — Lift magic numbers into named constants

In [ ]:
from dataclasses import dataclass
from collections import Counter
import math

### 1.1 — Constants (Task 05)
The bare `100` / `500` and the `"shipped"` magic string are lifted into named constants below — that *is* Task 05. As you write the functions, reference these names, **never raw literals**.

In [ ]:
# Task 05 -- magic numbers (and the magic status string) lifted into named constants.
MIN_VALID_AMOUNT: float = 0.0           # smallest valid amount (used in Part 2)
HIGH_VALUE_THRESHOLD: float = 100.0     # was the bare 100 in copy-paste block #1
PRIORITY_REVIEW_THRESHOLD: float = 500.0  # was the bare 500 in copy-paste block #3
SHIPPED: str = "shipped"                # the status string compared all over the legacy

### 1.2 — The `Order` dataclass (Task 02)
A typed, self-documenting record — the right tool for structured data.

In [ ]:
# Task 02 -- model the order record as a typed dataclass.
@dataclass
class Order:
    ...  # TODO: add four typed fields -> id: str, amount: float, status: str, customer: str

### 1.3 — `parse_orders`: the validation boundary (Tasks 03 + 04)

This is where the **bare except** goes to die. Instead of wrapping every field access in `try/except`, validate **structure once, at the boundary**, by constructing typed `Order` objects. If a raw record is malformed, construction fails **loudly here** — not silently mid-aggregation. Everything downstream can then trust that every `Order` has every field.

In [ ]:
def parse_orders(raw: list[dict]) -> list[Order]:
    """Build typed Order objects from raw dicts (the validation boundary)."""
    # TODO (Task 04): one list comprehension. Hint: Order(**record) unpacks a dict,
    # and raises loudly if a key is missing or extra.
    raise NotImplementedError("Implement parse_orders, then re-run.")

### 1.4 — Transform functions (Tasks 01 + 04)
`total_shipped` is a **worked example** — follow its shape for the other three.

In [ ]:
def total_shipped(orders: list[Order]) -> float:
    """Sum the amounts of all shipped orders. (Worked example -- follow this pattern.)"""
    return sum(o.amount for o in orders if o.status == SHIPPED)


def flag_high_value(orders: list[Order], threshold: float = HIGH_VALUE_THRESHOLD) -> list[str]:
    """Return the IDs of orders whose amount exceeds `threshold`."""
    # TODO (Tasks 01 + 04): one comprehension, like total_shipped above.
    raise NotImplementedError("Implement flag_high_value, then re-run.")


def count_by_status(orders: list[Order]) -> dict[str, int]:
    """Return a mapping of status -> number of orders with that status."""
    # TODO: replace the legacy manual-accumulation loop. collections.Counter helps.
    raise NotImplementedError("Implement count_by_status, then re-run.")


def priority_review(orders: list[Order], threshold: float = PRIORITY_REVIEW_THRESHOLD) -> list[str]:
    """Return IDs of SHIPPED orders above the priority threshold (note the AND)."""
    # TODO: combine a status check and an amount check in one comprehension.
    raise NotImplementedError("Implement priority_review, then re-run.")

### 1.5 — Smoke test (your *Done when*)
This cell fails until your functions are implemented. When it prints the green check, Part 1 is done: the refactor reproduces the legacy output exactly.

In [ ]:
orders = parse_orders(RAW_ORDERS)

# Floats are compared with math.isclose -- a good habit you'll use all Academy.
assert math.isclose(total_shipped(orders), legacy_total), "total_shipped mismatch"
assert flag_high_value(orders) == legacy_flagged, "flag_high_value mismatch"
assert count_by_status(orders) == legacy_counts, "count_by_status mismatch"
assert priority_review(orders) == legacy_priority, "priority_review mismatch"
print("\u2705 Smoke check passed -- refactor matches the legacy output.")

## Part 2 — Validation & stretch

**Stretch 1 — custom exception.** The bare except used to swallow bad data. Now make bad data **fail loudly and specifically**. **Stretch 2 — docstrings** on every function (you've got a head start).

In [ ]:
class InvalidOrderError(Exception):
    """Raised when an order is structurally valid but breaks a business rule."""
    def __init__(self, order_id: str, reason: str) -> None:
        self.order_id = order_id
        super().__init__(f"Order {order_id!r} is invalid: {reason}")


def validate_order(order: Order) -> None:
    """Raise InvalidOrderError if the order fails a business rule."""
    # TODO (Stretch 1): raise InvalidOrderError when amount < MIN_VALID_AMOUNT,
    # and again when status is empty. Give a helpful reason each time.
    raise NotImplementedError("Implement validate_order, then re-run.")

### 2.1 — Prove it rejects bad data
These records are deliberately broken. Good validation **rejects them loudly** — the exact opposite of the legacy bare except.

In [ ]:
dirty = [
    Order(id="ORD-9001", amount=-5.0, status="shipped", customer="Phantom Co"),  # negative
    Order(id="ORD-9002", amount=10.0, status="",        customer="Mystery LLC"), # empty status
]
for bad in dirty:
    try:
        validate_order(bad)
        print(f"{bad.id}: passed (unexpected!)")
    except InvalidOrderError as e:
        print(f"Rejected loudly -> {e}")

## Part 3 — Pull the day together 🧩

Everything from today converges here. The morning's **config object** (`LLMConfig`) and **client stub** (`call_model`) are brought forward. You'll build a prompt from your clean, typed orders and hand it to the model stub — the *"model as a dependency"* thesis, made concrete. No real API call today; that's Week 3.

In [ ]:
# Brought forward from this morning -- the config object + client stub.
@dataclass
class LLMConfig:
    model: str = "placeholder-model-id"  # real ID confirmed in Week 3 -- never hardcode
    temperature: float = 0.7
    max_tokens: int = 512
    timeout: float = 30.0


def call_model(prompt: str, cfg: LLMConfig) -> str:
    """Deterministic STUB. The real API call lands in Week 3."""
    return f"[STUB] model={cfg.model} | temp={cfg.temperature} | prompt_chars={len(prompt)}"

In [ ]:
def build_review_prompt(orders: list[Order]) -> str:
    """Pure function: turn the priority-review orders into a prompt for the model."""
    priority_ids = set(priority_review(orders))
    # TODO (Task 04, new context): build one line per priority order, e.g.
    #   "- ORD-1003: $980.00 (Oakfield Renovations)"
    # using a comprehension over orders whose id is in priority_ids, then join with newlines.
    raise NotImplementedError("Implement build_review_prompt, then re-run.")

In [ ]:
cfg = LLMConfig()
prompt = build_review_prompt(orders)
response = call_model(prompt, cfg)
print(response)

# Responsible-AI gesture: even a stub gets a basic sanity check before we trust it.
# In Week 3 this becomes a real call; in Week 9 this assert grows into output guardrails.
assert response, "model returned an empty response"
print("\u2705 Day pulled together: typed data -> clean functions -> a model in the loop.")

## Close — Restart & Run All

**Kernel → Restart Kernel and Run All Cells.** Every cell must execute top-to-bottom with no errors before this counts as done.

✅ **Done when:** smoke check passes · no bare except anywhere · every function is typed and has a docstring · at least one comprehension replaced an accumulator loop · magic numbers are named constants · bad data is rejected loudly · the Part 3 synthesis prints its green check.

**Before you leave, be ready to answer:**
1. What did *Restart-and-Run-All* protect you from?
2. Where does a class beat a function here — and where is it overkill?
3. Why store the model ID in a config object instead of inline?